<div style="font-size:2em; font-weight:bold; margin-bottom:8px;">05 — Generate Embeddings</div>

This notebook reads the enriched chunks produced by notebook 04 and turns each chunk's text into a numeric **embedding vector** using a free, local model.

It is **Step 5** of the RAG data indexing pipeline — embedding only.

---

**What this notebook does:**
1. Loads the enriched chunks from `data/processed/04_chunks_with_metadata.jsonl`
2. Loads the local embedding model `BAAI/bge-small-en-v1.5` and confirms the vector size
3. Embeds a single chunk to show the vector shape
4. Embeds every chunk's text in batches
5. Sanity-checks the vectors (count, no NaNs, unit norms)
6. Saves the vectors to `data/processed/05_embeddings.npy` and the aligned metadata to `05_embeddings_meta.jsonl`
7. Reloads both files and verifies they line up

**What this notebook intentionally does NOT do:**
- No Qdrant indexing (that is notebook 06)
- No paid embedding API — everything runs locally and for free

> **Before running:** make sure dependencies are installed.
> ```bash
> pip install -r requirements.txt
> ```
> The first run downloads the model (~130 MB) and caches it for later runs.

---
## 1. Imports

We need:
- **`json`** — read the enriched chunks and write the aligned metadata
- **`numpy`** — store the embedding matrix efficiently in a `.npy` file
- **`pathlib.Path`** — OS-independent file paths
- **`torch`** — used only to detect whether a GPU is available
- **`SentenceTransformer`** — loads and runs the local embedding model

In [1]:
# ── [1 / 10] Imports ────────────────────────────────────────────────────────

import json
from pathlib import Path

import numpy as np
import torch
from sentence_transformers import SentenceTransformer

print("Imports ready.")

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports ready.


---
## 2. Configuration — Model, Batch Size, Device, and Paths

| Setting | Meaning |
|---|---|
| `EMBEDDING_MODEL` | The free local model that turns text into vectors |
| `BATCH_SIZE` | How many chunks to embed at once (larger = faster, more memory) |
| `NORMALIZE` | Normalize vectors to unit length so cosine similarity works cleanly |
| `DEVICE` | Auto-detected: `cuda` if a GPU is available, `cpu` otherwise |

We save two aligned files so row `i` of the vector matrix matches line `i` of the
metadata file:
- `05_embeddings.npy` — the dense vector matrix
- `05_embeddings_meta.jsonl` — the chunk metadata (no vectors), one per line

In [2]:
# ── [2 / 10] Configuration ──────────────────────────────────────────────────

EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
BATCH_SIZE      = 64
NORMALIZE       = True

# Automatically use GPU if available, otherwise fall back to CPU.
# No manual configuration needed.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Resolve the project root no matter where the notebook is run from
_cwd = Path.cwd()
ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd

# Input — produced by notebook 04
INPUT_FILE = ROOT / "data" / "processed" / "04_chunks_with_metadata.jsonl"

# Output — vectors + aligned metadata for the indexing step
OUTPUT_DIR    = ROOT / "data" / "processed"
VECTORS_FILE  = OUTPUT_DIR / "05_embeddings.npy"
META_FILE     = OUTPUT_DIR / "05_embeddings_meta.jsonl"

print(f"Project root : {ROOT}")
print(f"Input file   : {INPUT_FILE}")
print(f"Vectors file : {VECTORS_FILE}")
print(f"Meta file    : {META_FILE}")
print(f"Input exists : {INPUT_FILE.exists()}")
print(f"Device       : {DEVICE}")

Project root : /app
Input file   : /app/data/processed/04_chunks_with_metadata.jsonl
Vectors file : /app/data/processed/05_embeddings.npy
Meta file    : /app/data/processed/05_embeddings_meta.jsonl
Input exists : True
Device       : cuda


---
## 3. Load the Enriched Chunks

We read the JSONL file produced by notebook 04. We will embed the `text` field and
keep all the other fields as the aligned metadata.

In [3]:
# ── [3 / 10] Load the Enriched Chunks ───────────────────────────────────────

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Input file not found: {INPUT_FILE}\n"
        "Run notebook 04_enrich_metadata.ipynb first."
    )

records = []
with INPUT_FILE.open("r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

texts = [r["text"] for r in records]

print(f"Loaded {len(records):,} chunks")
print(f"First text preview: {texts[0][:100]}...")

Loaded 12,281 chunks
First text preview: Alterations of the architecture of cerebral white matter in the developing human brain can affect co...


---
## 4. Load the Embedding Model

We load `BAAI/bge-small-en-v1.5` with `sentence-transformers`. The first run
downloads the model; later runs load it from cache. We also read the model's
vector dimension — for this model it is **384**, which is the vector size Qdrant
will need in notebook 06.

In [4]:
# ── [4 / 10] Load the Embedding Model ───────────────────────────────────────

print(f"Loading model : {EMBEDDING_MODEL}")
print(f"Device        : {DEVICE}")
model = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)

vector_size = model.get_embedding_dimension()
print(f"Model loaded. Vector dimension = {vector_size}")

Loading model : BAAI/bge-small-en-v1.5
Device        : cuda


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1355.62it/s]


Model loaded. Vector dimension = 384


---
## 5. Embed a Single Chunk

Before embedding thousands of chunks, we embed one to see the output: a 1D array
of `vector_size` floating-point numbers.

In [5]:
# ── [5 / 10] Embed a Single Chunk ───────────────────────────────────────────

sample_vector = model.encode(texts[0], normalize_embeddings=NORMALIZE)

print(f"Vector shape : {sample_vector.shape}")
print(f"Vector dtype : {sample_vector.dtype}")
print(f"First 8 values: {np.round(sample_vector[:8], 4)}")
print(f"L2 norm      : {np.linalg.norm(sample_vector):.4f}  (≈1.0 when normalized)")

Vector shape : (384,)
Vector dtype : float32
First 8 values: [-0.0466 -0.083  -0.0054  0.0114  0.0243  0.0713  0.003   0.0068]
L2 norm      : 1.0000  (≈1.0 when normalized)


---
## 6. Embed Every Chunk

Now we embed all chunk texts in batches. `sentence-transformers` handles batching
internally; `show_progress_bar=True` prints progress so you can watch it run.

The device was selected automatically in the Configuration cell: **GPU** when
available (much faster), **CPU** as fallback (~5 minutes for the full SciFact
corpus on CPU — that is expected).

In [6]:
# ── [6 / 10] Embed Every Chunk ──────────────────────────────────────────────

embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    normalize_embeddings=NORMALIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
)

# Store as float32 — half the memory of float64 and what Qdrant expects
embeddings = embeddings.astype(np.float32)

print()
print(f"Embeddings shape : {embeddings.shape}  (rows = chunks, cols = vector size)")
print(f"Embeddings dtype : {embeddings.dtype}")

Batches: 100%|██████████| 192/192 [00:10<00:00, 18.98it/s]


Embeddings shape : (12281, 384)  (rows = chunks, cols = vector size)
Embeddings dtype : float32


---
## 7. Sanity-Check the Vectors

Before saving, we confirm:
- The number of vectors equals the number of chunks (alignment).
- There are no `NaN` or infinite values.
- When normalization is on, every vector has length ≈ 1.0.

In [7]:
# ── [7 / 10] Sanity-Check the Vectors ───────────────────────────────────────

n_vectors, n_dims = embeddings.shape

has_nan = bool(np.isnan(embeddings).any())
has_inf = bool(np.isinf(embeddings).any())
norms   = np.linalg.norm(embeddings, axis=1)

print(f"Vectors        : {n_vectors:,}")
print(f"Chunks         : {len(records):,}")
print(f"Dimensions     : {n_dims}")
print(f"Contains NaN   : {has_nan}")
print(f"Contains Inf   : {has_inf}")
print(f"Norm min/max   : {norms.min():.4f} / {norms.max():.4f}")
print()

assert n_vectors == len(records), "vector count does not match chunk count"
assert not has_nan, "embeddings contain NaN values"
assert not has_inf, "embeddings contain infinite values"
if NORMALIZE:
    assert np.allclose(norms, 1.0, atol=1e-3), "normalized vectors should have norm ≈ 1.0"
print("Vectors verified — aligned, finite, and correctly normalized.")

Vectors        : 12,281
Chunks         : 12,281
Dimensions     : 384
Contains NaN   : False
Contains Inf   : False
Norm min/max   : 1.0000 / 1.0000

Vectors verified — aligned, finite, and correctly normalized.


---
## 8. Save Vectors and Aligned Metadata

We save two files that share the same row order:
- `05_embeddings.npy` — the vector matrix (via `numpy.save`)
- `05_embeddings_meta.jsonl` — one metadata record per line (no vectors)

Keeping vectors and metadata in separate, aligned files keeps the `.npy` file
small and fast to load while the metadata stays human-readable.

In [8]:
# ── [8 / 10] Save Vectors and Aligned Metadata ──────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save the vector matrix
np.save(VECTORS_FILE, embeddings)

# Save the aligned metadata (same order as the vectors)
with META_FILE.open("w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Saved vectors  : {VECTORS_FILE.resolve()}  shape={embeddings.shape}")
print(f"Saved metadata : {META_FILE.resolve()}  lines={len(records):,}")

Saved vectors  : /app/data/processed/05_embeddings.npy  shape=(12281, 384)
Saved metadata : /app/data/processed/05_embeddings_meta.jsonl  lines=12,281


---
## 9. Reload and Verify Alignment

We reload both files and confirm the vector matrix and the metadata still have the
same number of rows and that the dimension matches the model.

In [9]:
# ── [9 / 10] Reload and Verify Alignment ────────────────────────────────────

reloaded_vectors = np.load(VECTORS_FILE)

reloaded_meta = []
with META_FILE.open("r", encoding="utf-8") as f:
    for line in f:
        reloaded_meta.append(json.loads(line))

print(f"Reloaded vectors shape : {reloaded_vectors.shape}")
print(f"Reloaded meta lines    : {len(reloaded_meta):,}")
print()

assert reloaded_vectors.shape[0] == len(reloaded_meta), "vectors and metadata are misaligned"
assert reloaded_vectors.shape[1] == vector_size,        "vector dimension mismatch"
print("Alignment verified — vectors and metadata match row for row.")

Reloaded vectors shape : (12281, 384)
Reloaded meta lines    : 12,281

Alignment verified — vectors and metadata match row for row.


---
## 10. Final Summary

A short summary of what this notebook produced, ready for the indexing step in
notebook 06.

In [10]:
# ── [10 / 10] Final Summary ─────────────────────────────────────────────────

print("=" * 60)
print("Embedding step complete")
print("=" * 60)
print(f"  Model         : {EMBEDDING_MODEL}")
print(f"  Vectors       : {reloaded_vectors.shape[0]:,}")
print(f"  Dimension     : {reloaded_vectors.shape[1]}")
print(f"  Normalized    : {NORMALIZE}")
print(f"  Vectors file  : {VECTORS_FILE.name}")
print(f"  Metadata file : {META_FILE.name}")
print()
print("Next: run 06_index_qdrant.ipynb to store these vectors in Qdrant.")

Embedding step complete
  Model         : BAAI/bge-small-en-v1.5
  Vectors       : 12,281
  Dimension     : 384
  Normalized    : True
  Vectors file  : 05_embeddings.npy
  Metadata file : 05_embeddings_meta.jsonl

Next: run 06_index_qdrant.ipynb to store these vectors in Qdrant.
